# 01 — Explore
First pass over the data: shapes, types, nulls, odd values.

Two ways in:
- `load_table(...)` — raw CSV straight into pandas (bypasses dbt). Good for profiling.
- `connect()` / `read(...)` — the dbt-built warehouse, read-only. Run `uv run ingest`, then `cd dbt; uv run dbt build` first.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import duckdb
from datawings import load_all, connect, read

pd.set_option("display.max_columns", 50)
dfs = load_all()
{k: v.shape for k, v in dfs.items()}

## Profile the raw tables

In [ ]:
for name, df in dfs.items():
    print(f"\n== {name} {df.shape}")
    print(df.isna().sum()[lambda s: s > 0].to_string() or 'no nulls')

## What's in the warehouse

In [ ]:
con = connect()  # read-only. Close it (last cell) or restart the kernel before `dbt build`.
con.sql("select table_schema, table_name from information_schema.tables order by 1, 2").df()

## pandas vs DuckDB — warm-up
Flights by status, three ways.

In [ ]:
dfs["flight"]["status"].value_counts()

In [ ]:
con.sql("select status, count(*) as n from silver.flight group by all order by n desc").df()

DuckDB can also query a pandas DataFrame in scope directly:

In [ ]:
crew = dfs["crew"]
duckdb.sql("select rank, count(*) n from crew group by all order by n desc").df()

## Questions to chase
- How clean is `crew.rank`? Does `ref.rank_map` cover every variant?
- Does the qualification `status` flag agree with the dates?
- Actual vs planned FDP — how many duties breach the MAX_FDP limit?
- Rolling 7d duty / 28d block hours per crew — who gets close to the limits?
- Disruptions: what triggers them, how are they recovered, what does it cost in delay?

In [ ]:
con.close()  # release the file before running dbt again